In [ ]:
import os
import zipfile
import pickle
import numpy as np
import tensorflow as tf
from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from keras.models import Model
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from keras.losses import BinaryFocalCrossentropy

colors_segTHRawS = ['black', 'red']
my_cmap_segTHRawS = ListedColormap(colors_segTHRawS)

box_0_segTHRawS = mpatches.Patch(color=colors_segTHRawS[0], label='0: Background')
box_1_segTHRawS = mpatches.Patch(color=colors_segTHRawS[1], label='1: Burned area')

bewerkte_data_dir = '/home/laura/Scriptie/bewerkte_data/segTHRawS'

In [ ]:
X_dataset = np.load(os.path.join(bewerkte_data_dir, 'X_train_segTHRawS.npy'))
y_dataset = np.load(os.path.join(bewerkte_data_dir, 'y_train_segTHRawS.npy'))

X_train, X_val, y_train, y_val = train_test_split(X_dataset, y_dataset, test_size=0.2, shuffle=False)

max_value = np.max(X_train)
X_train_unet = X_train.astype('float32') / max_value
X_val_unet = X_val.astype('float32') / max_value

# Unet architecture
def build_unet(input_shape=(256, 256, 3), num_classes=2):
    inputs = Input(input_shape)
    
    # Encoder
    c1 = Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
    p1 = MaxPooling2D((2, 2))(c1)
    
    c2 = Conv2D(32, (3, 3), activation='relu', padding='same')(p1)
    p2 = MaxPooling2D((2, 2))(c2)
    
    # Bottleneck
    c3 = Conv2D(64, (3, 3), activation='relu', padding='same')(p2)
    
    # Decoder
    u4 = UpSampling2D((2, 2))(c3)
    concat4 = concatenate([u4, c2])
    c4 = Conv2D(32, (3, 3), activation='relu', padding='same')(concat4)
    
    u5 = UpSampling2D((2, 2))(c4)
    concat5 = concatenate([u5, c1]) 
    c5 = Conv2D(16, (3, 3), activation='relu', padding='same')(concat5)
    
    # output layer (num_classes = 1)
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c5)
    
    model = Model(inputs=[inputs], outputs=[outputs])
    return model

unet = build_unet()
unet.compile(optimizer='adam', loss=BinaryFocalCrossentropy(alpha=0.9, gamma=2.0), metrics=['accuracy'])

In [ ]:
unet_history = unet.fit(
    X_train, y_train,
    batch_size=4,
    epochs=5, 
    validation_data=(X_val, y_val), 
    verbose=1
)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(unet_history.history['loss'], label='Training Loss')
plt.plot(unet_history.history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(unet_history.history['accuracy'], label='Training Accuracy')
plt.plot(unet_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 50% chance of fire per pixel
X_visual = X_val_unet[0:1]
y_visual_true = y_val[0]

y_visual_pred = unet.predict(X_visual)
y_visual_pred_class = np.argmax(y_visual_pred, axis=-1)  

predicted_class = y_visual_pred_class[0]  
actual_class = y_visual_true[..., 0] if len(y_visual_true.shape) == 3 else y_visual_true

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(predicted_class, cmap=my_cmap_segTHRawS, vmin=0, vmax=1)
plt.title('Predicted Class (U-Net)')
plt.legend(handles=[box_0_segTHRawS, box_1_segTHRawS], bbox_to_anchor=(1.05, 1), loc='upper left')

plt.subplot(1, 2, 2)
plt.imshow(actual_class, cmap=my_cmap_segTHRawS, vmin=0, vmax=1)
plt.title('Actual Class (Realiteit)')
plt.legend(handles=[box_0_segTHRawS, box_1_segTHRawS], bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# 5% chance of fire per pixel
X_visual = X_val_unet[0:1]
y_visual_true = y_val[0]

y_visual_pred = unet.predict(X_visual)

fire_probabilities = y_visual_pred[0, :, :, 0]

actual_class = y_visual_true[..., 0] if len(y_visual_true.shape) == 3 else y_visual_true

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
lower_threshold = (fire_probabilities > 0.05).astype(int)
plt.imshow(lower_threshold, cmap=my_cmap_segTHRawS, vmin=0, vmax=1)
plt.title('Predicted Class (U-Net) - 5% Threshold')

plt.subplot(1, 3, 2)
plt.imshow(fire_probabilities, cmap='hot', vmin=0, vmax=1)
plt.title('Heatmap')
plt.colorbar(fraction=0.046, pad=0.04) 

plt.subplot(1, 3, 3)
plt.imshow(actual_class, cmap=my_cmap_segTHRawS, vmin=0, vmax=1)
plt.title('Actual Class')

plt.tight_layout()
plt.show()

In [ ]:
bewerkte_data_dir = '/home/laura/Scriptie/bewerkte_data/segTHRawS'

all_unet_preds = unet.predict(X_val_unet, batch_size=4)

unet_probs = all_unet_preds[..., 0]
unet_predictions = (unet_probs > 0.5).astype(int)

np.save(os.path.join(bewerkte_data_dir, 'y_pred_probs_fire_unet.npy'), unet_probs)
np.save(os.path.join(bewerkte_data_dir, 'y_pred_masks_unet.npy'), unet_predictions)
np.save(os.path.join(bewerkte_data_dir, 'y_test_masks_unet.npy'), y_val)

Here, we can look into dice loss, zodat de F1 score verbeterd kan worden. Heb ik op dit moment geen tijd voor, maar mocht die tijd er wel zijn..